# Graph Attention Networks (GAT) — Colab

This notebook mirrors `code/model.py`, `code/train.py`, and `code/evaluate.py` in the repo: **2-layer GAT** on **Cora** / **CiteSeer** with 100 runs and **mean ± std** test accuracy (paper Table 2 targets: **83.0 ± 0.7%** / **72.5 ± 0.7%**).

## Colab setup

1. **Runtime → Change runtime type →** enable **GPU** (recommended; 100 full runs on CPU is slow).
2. Run the **Install** cell below once per runtime.
3. Set `NUM_RUNS` in the last section (use `3` for a quick smoke test, `100` to match the paper’s protocol).
4. Optional: mount Drive and set `PLANETOID_PARENT` / `RESULTS_DIR` to a folder under `/content/drive/MyDrive/...` so data and CSVs survive disconnects.

In [ ]:
# @title Install dependencies (run once per Colab session)
!pip install -q torch-geometric

In [ ]:
# @title Config: paths and imports
import csv
import pathlib
import sys
from typing import List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GATConv
from torch_geometric.transforms import NormalizeFeatures

# Planetoid download/cache. Under /content so it persists for the Colab session.
PLANETOID_PARENT = pathlib.Path("/content/gat_data")
RESULTS_DIR = pathlib.Path("/content/gat_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(), "| cwd:", pathlib.Path().resolve())

In [ ]:
# @title Model (same as `code/model.py`)
class ConfigurableGAT(nn.Module):
    def __init__(self, num_features, num_classes, hidden_channels, heads_first, heads_second=1, dropout=0.6):
        super().__init__()
        self.dropout = dropout
        self.conv1 = GATConv(
            in_channels=num_features,
            out_channels=hidden_channels,
            heads=heads_first,
            dropout=dropout,
            concat=True,
        )
        self.conv2 = GATConv(
            in_channels=hidden_channels * heads_first,
            out_channels=num_classes,
            heads=heads_second,
            dropout=dropout,
            concat=False,
        )

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)

    def forward_with_attention(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x, attn1 = self.conv1(x, edge_index, return_attention_weights=True)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x, attn2 = self.conv2(x, edge_index, return_attention_weights=True)
        return F.log_softmax(x, dim=1), attn1, attn2


class GAT(ConfigurableGAT):
    def __init__(self, num_features, num_classes, dropout=0.6):
        super().__init__(num_features, num_classes,
                         hidden_channels=8, heads_first=8, heads_second=1, dropout=dropout)


class TinyGAT(ConfigurableGAT):
    def __init__(self, num_features, num_classes, dropout=0.6):
        super().__init__(num_features, num_classes,
                         hidden_channels=4, heads_first=2, heads_second=1, dropout=dropout)


In [ ]:
# @title Training (same as `code/train.py`)
LR = 0.005
WEIGHT_DECAY = 5e-4
DROPOUT = 0.6
EPOCHS = 10_000
PATIENCE = 100


def train_epoch(model, data, optimizer):
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()


@torch.no_grad()
def eval_split(model, data):
    model.eval()
    out = model(data.x, data.edge_index)
    val_loss = F.nll_loss(out[data.val_mask], data.y[data.val_mask]).item()
    pred = out.argmax(dim=1)
    val_acc = (pred[data.val_mask] == data.y[data.val_mask]).float().mean().item()
    test_acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()
    return val_loss, val_acc, test_acc


def run(
    dataset_name: str,
    seed: int = 42,
    data_parent: Optional[pathlib.Path] = None,
) -> float:
    torch.manual_seed(seed)
    parent = data_parent or PLANETOID_PARENT
    parent.mkdir(parents=True, exist_ok=True)
    root = str(parent / dataset_name)

    dataset = Planetoid(
        root=root,
        name=dataset_name,
        transform=NormalizeFeatures(),
    )
    data = dataset[0]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    data = data.to(device)

    model = GAT(
        num_features=dataset.num_features,
        num_classes=dataset.num_classes,
        dropout=DROPOUT,
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best_val_loss = float("inf")
    best_test_acc = 0.0
    patience_counter = 0

    for _epoch in range(1, EPOCHS + 1):
        train_epoch(model, data, optimizer)
        val_loss, _val_acc, test_acc = eval_split(model, data)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_test_acc = test_acc
            patience_counter = 0
        else:
            patience_counter += 1
        if patience_counter >= PATIENCE:
            break

    return best_test_acc

In [ ]:
# @title Multi-run evaluation (same as `code/evaluate.py`)
def run_many(dataset_name: str, num_runs: int) -> Tuple[float, float, pathlib.Path]:
    csv_path = RESULTS_DIR / f"{dataset_name}_results.csv"
    accuracies: List[float] = []
    with open(csv_path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["run", "test_accuracy"])
        for i in range(1, num_runs + 1):
            acc = run(dataset_name, seed=i)
            accuracies.append(acc)
            w.writerow([i, f"{acc:.6f}"])
            f.flush()
            sys.stdout.flush()
            print(f"Run {i:3d}/{num_runs}  test_acc={acc * 100:.2f}%", flush=True)
    mean_acc = float(np.mean(accuracies) * 100)
    std_acc = float(np.std(accuracies) * 100)
    print("\n" + "=" * 45)
    print(f"Dataset : {dataset_name}")
    print(f"Runs    : {num_runs}")
    print(f"Mean    : {mean_acc:.2f}%")
    print(f"Std     : {std_acc:.2f}%")
    print(f"Results saved to: {csv_path}")
    print("=" * 45)
    return mean_acc, std_acc, csv_path

In [ ]:
# @title Run experiments
# Set NUM_RUNS=100 to match the paper; use 3–5 for a quick test.
NUM_RUNS = 100
DATASETS = ("Cora", "CiteSeer")
summary = {}
for name in DATASETS:
    m, s, _p = run_many(name, num_runs=NUM_RUNS)
    summary[name] = (m, s)
print("\nSummary (mean, std) %:")
for k, (m, s) in summary.items():
    print(f"  {k}: {m:.2f} ± {s:.2f}")

## Fast Extensions (few-days Colab plan)

This section adds:
1. **Low-Label Regime Benchmark**
2. **TinyGAT Distillation**
3. **Failure Case Explorer**

Recommended defaults for limited time:
- Cora first
- `runs=3` for smoke test, then `runs=5-10`
- Keep `student_epochs` modest (e.g., `1000-2000`)

In [ ]:
# @title Extensions setup: flexible runner
import json
from collections import Counter


def load_planetoid(dataset_name: str):
    root = str((PLANETOID_PARENT / dataset_name))
    dataset = Planetoid(root=root, name=dataset_name, transform=NormalizeFeatures())
    data = dataset[0]
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    return dataset, data.to(device), device


def build_model(model_name: str, dataset, dropout=DROPOUT):
    if model_name == 'gat':
        return GAT(dataset.num_features, dataset.num_classes, dropout=dropout)
    if model_name == 'tinygat':
        return TinyGAT(dataset.num_features, dataset.num_classes, dropout=dropout)
    raise ValueError(f'Unknown model_name={model_name}')


def run_model(dataset_name: str, seed: int = 42, model_name: str = 'gat', train_mask_override=None, return_model=False):
    torch.manual_seed(seed)
    dataset, data, device = load_planetoid(dataset_name)

    if train_mask_override is not None:
        data.train_mask = train_mask_override.to(device)

    model = build_model(model_name, dataset, dropout=DROPOUT).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best_val_loss = float('inf')
    best_test_acc = 0.0
    patience_counter = 0

    for _ in range(1, EPOCHS + 1):
        train_epoch(model, data, optimizer)
        val_loss, _, test_acc = eval_split(model, data)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_test_acc = test_acc
            patience_counter = 0
        else:
            patience_counter += 1
        if patience_counter >= PATIENCE:
            break

    if return_model:
        return best_test_acc, model, data
    return best_test_acc


print('Extensions setup loaded.')


In [ ]:
# @title 1) Low-Label Regime Benchmark
import pandas as pd


def build_low_label_mask(base_train_mask: torch.Tensor, ratio: float, seed: int):
    idx = torch.where(base_train_mask.cpu())[0]
    keep = max(1, int(len(idx) * ratio))
    g = torch.Generator(device='cpu')
    g.manual_seed(seed)
    perm = torch.randperm(len(idx), generator=g)
    chosen = idx[perm[:keep]]
    mask = torch.zeros_like(base_train_mask.cpu(), dtype=torch.bool)
    mask[chosen] = True
    return mask


def low_label_benchmark(dataset_name='Cora', ratios=(0.1, 0.2, 0.5, 1.0), runs=5, model_name='gat'):
    dataset, data, _ = load_planetoid(dataset_name)
    base_mask = data.train_mask.detach().cpu()
    base_count = int(base_mask.sum().item())

    rows = []
    for ratio in ratios:
        print(f'\nratio={ratio:.2f}')
        for seed in range(1, runs + 1):
            m = build_low_label_mask(base_mask, ratio=ratio, seed=seed)
            acc = run_model(dataset_name, seed=seed, model_name=model_name, train_mask_override=m)
            rows.append({
                'dataset': dataset_name,
                'model': model_name,
                'ratio': ratio,
                'run': seed,
                'num_train_labels': int(m.sum().item()),
                'base_num_train_labels': base_count,
                'test_accuracy': acc,
            })
            print(f'  run {seed}/{runs}: {acc*100:.2f}%')

    df = pd.DataFrame(rows)
    summary = (
        df.groupby(['dataset', 'model', 'ratio', 'num_train_labels'], as_index=False)
          .agg(mean_test_accuracy=('test_accuracy', 'mean'), std_test_accuracy=('test_accuracy', 'std'))
    )
    summary['std_test_accuracy'] = summary['std_test_accuracy'].fillna(0.0)

    detail_path = RESULTS_DIR / f'{dataset_name}_low_label_detail_colab.csv'
    summary_path = RESULTS_DIR / f'{dataset_name}_low_label_summary_colab.csv'
    df.to_csv(detail_path, index=False)
    summary.to_csv(summary_path, index=False)

    print(f'\nSaved detail : {detail_path}')
    print(f'Saved summary: {summary_path}')
    return df, summary


# Few-days default (fast):
LOW_LABEL_DATASET = 'Cora'   # change to 'CiteSeer' after Cora
LOW_LABEL_RUNS = 5           # use 3 for smoke test, 10 for better estimate
LOW_LABEL_RATIOS = (0.1, 0.2, 0.5, 1.0)

low_label_df, low_label_summary = low_label_benchmark(
    dataset_name=LOW_LABEL_DATASET,
    ratios=LOW_LABEL_RATIOS,
    runs=LOW_LABEL_RUNS,
    model_name='gat',
)
low_label_summary

In [ ]:
# @title 2) TinyGAT Distillation (teacher GAT -> student TinyGAT)

def train_tiny_student(student, data, teacher_log_probs=None, alpha=0.7, temperature=2.0, max_epochs=1500):
    opt = torch.optim.Adam(student.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    best_val_loss = float('inf')
    best_test_acc = 0.0
    patience_counter = 0

    for _ in range(max_epochs):
        student.train()
        opt.zero_grad()
        student_log_probs = student(data.x, data.edge_index)

        hard_loss = F.nll_loss(student_log_probs[data.train_mask], data.y[data.train_mask])
        if teacher_log_probs is None:
            loss = hard_loss
        else:
            s_log_t = F.log_softmax(student_log_probs / temperature, dim=1)
            t_prob_t = F.softmax(teacher_log_probs / temperature, dim=1)
            soft_loss = F.kl_div(
                s_log_t[data.train_mask],
                t_prob_t[data.train_mask],
                reduction='batchmean',
            ) * (temperature ** 2)
            loss = alpha * soft_loss + (1 - alpha) * hard_loss

        loss.backward()
        opt.step()

        val_loss, _, test_acc = eval_split(student, data)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_test_acc = test_acc
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            break

    return best_test_acc


@torch.no_grad()
def teacher_logits(model, data):
    model.eval()
    return model(data.x, data.edge_index)


def distill_benchmark(dataset_name='Cora', runs=5, alpha=0.7, temperature=2.0, student_epochs=1500):
    rows = []

    for seed in range(1, runs + 1):
        teacher_acc, teacher_model, teacher_data = run_model(
            dataset_name,
            seed=seed,
            model_name='gat',
            return_model=True,
        )
        t_log_probs = teacher_logits(teacher_model, teacher_data)

        dataset, data, device = load_planetoid(dataset_name)
        tiny_sup = build_model('tinygat', dataset).to(device)
        tiny_distill = build_model('tinygat', dataset).to(device)

        tiny_sup_acc = train_tiny_student(
            tiny_sup,
            data,
            teacher_log_probs=None,
            alpha=alpha,
            temperature=temperature,
            max_epochs=student_epochs,
        )
        tiny_distill_acc = train_tiny_student(
            tiny_distill,
            data,
            teacher_log_probs=t_log_probs,
            alpha=alpha,
            temperature=temperature,
            max_epochs=student_epochs,
        )

        gain = tiny_distill_acc - tiny_sup_acc
        rows.append({
            'dataset': dataset_name,
            'seed': seed,
            'teacher_acc': teacher_acc,
            'tiny_supervised_acc': tiny_sup_acc,
            'tiny_distilled_acc': tiny_distill_acc,
            'distill_gain': gain,
        })
        print(
            f"seed={seed} | teacher={teacher_acc*100:.2f}% | "
            f"tiny_sup={tiny_sup_acc*100:.2f}% | tiny_distill={tiny_distill_acc*100:.2f}% | gain={gain*100:.2f}%"
        )

    distill_df = pd.DataFrame(rows)
    out_path = RESULTS_DIR / f'{dataset_name}_tinygat_distill_colab.csv'
    distill_df.to_csv(out_path, index=False)
    print(f'\nSaved: {out_path}')
    print(f"Mean distill gain: {distill_df['distill_gain'].mean()*100:.2f}%")
    return distill_df


# Few-days default (fast):
DISTILL_DATASET = 'Cora'   # then run on CiteSeer
DISTILL_RUNS = 5
DISTILL_ALPHA = 0.7
DISTILL_TEMP = 2.0
DISTILL_STUDENT_EPOCHS = 1500

distill_df = distill_benchmark(
    dataset_name=DISTILL_DATASET,
    runs=DISTILL_RUNS,
    alpha=DISTILL_ALPHA,
    temperature=DISTILL_TEMP,
    student_epochs=DISTILL_STUDENT_EPOCHS,
)
distill_df

In [ ]:
# @title 3) Failure Case Explorer

def failure_case_explorer(dataset_name='Cora', seed=1, max_cases=25, top_k_edges=5):
    test_acc, model, data = run_model(dataset_name, seed=seed, model_name='gat', return_model=True)
    model.eval()

    log_probs, attn1, _attn2 = model.forward_with_attention(data.x, data.edge_index)
    pred = log_probs.argmax(dim=1)
    conf = log_probs.exp().max(dim=1).values

    test_idx = torch.where(data.test_mask)[0]
    wrong_nodes = test_idx[pred[test_idx] != data.y[test_idx]]

    edge_index_1, alpha_1 = attn1
    alpha_1_mean = alpha_1.mean(dim=1)

    cases = []
    for node_id in wrong_nodes[:max_cases].tolist():
        incoming = edge_index_1[1] == node_id
        incoming_src = edge_index_1[0][incoming]
        incoming_alpha = alpha_1_mean[incoming]

        hist = Counter(data.y[incoming_src].tolist())
        neighbor_class_hist = {int(k): int(v) for k, v in sorted(hist.items())}

        top_edges = []
        if incoming_alpha.numel() > 0:
            k = min(top_k_edges, incoming_alpha.numel())
            top_vals, top_pos = torch.topk(incoming_alpha, k=k)
            src_nodes = incoming_src[top_pos]
            for src_node, alpha_val in zip(src_nodes.tolist(), top_vals.tolist()):
                top_edges.append({
                    'src_node': int(src_node),
                    'src_label': int(data.y[src_node].item()),
                    'attention': float(alpha_val),
                })

        cases.append({
            'node_id': int(node_id),
            'true_label': int(data.y[node_id].item()),
            'pred_label': int(pred[node_id].item()),
            'confidence': float(conf[node_id].item()),
            'neighbor_class_hist': neighbor_class_hist,
            'top_incoming_attention_edges': top_edges,
        })

    out_json = RESULTS_DIR / f'{dataset_name}_seed{seed}_failure_cases_colab.json'
    out_csv = RESULTS_DIR / f'{dataset_name}_seed{seed}_failure_cases_colab.csv'

    with open(out_json, 'w') as f:
        json.dump({
            'dataset': dataset_name,
            'seed': seed,
            'test_accuracy': test_acc,
            'num_cases': len(cases),
            'cases': cases,
        }, f, indent=2)

    flat_rows = []
    for c in cases:
        flat_rows.append({
            'node_id': c['node_id'],
            'true_label': c['true_label'],
            'pred_label': c['pred_label'],
            'confidence': c['confidence'],
            'neighbor_class_hist': json.dumps(c['neighbor_class_hist']),
            'top_incoming_attention_edges': json.dumps(c['top_incoming_attention_edges']),
        })
    pd.DataFrame(flat_rows).to_csv(out_csv, index=False)

    print(f'Test accuracy: {test_acc*100:.2f}%')
    print(f'Cases exported: {len(cases)}')
    print('JSON:', out_json)
    print('CSV :', out_csv)

    return pd.DataFrame(flat_rows), cases


# Few-days default (fast):
FAIL_DATASET = 'Cora'
FAIL_SEED = 1
FAIL_MAX_CASES = 25
FAIL_TOP_K_EDGES = 5

failure_df, failure_cases = failure_case_explorer(
    dataset_name=FAIL_DATASET,
    seed=FAIL_SEED,
    max_cases=FAIL_MAX_CASES,
    top_k_edges=FAIL_TOP_K_EDGES,
)
failure_df.head(10)

In [ ]:
# @title Quick plots for report-ready figures
import matplotlib.pyplot as plt


# Low-label curve
if 'low_label_summary' in globals() and len(low_label_summary) > 0:
    plt.figure(figsize=(6, 4))
    x = low_label_summary['ratio'].values
    y = low_label_summary['mean_test_accuracy'].values * 100
    yerr = low_label_summary['std_test_accuracy'].values * 100
    plt.errorbar(x, y, yerr=yerr, marker='o', capsize=4)
    plt.xlabel('Labeled training fraction')
    plt.ylabel('Test accuracy (%)')
    plt.title(f'Low-label benchmark ({LOW_LABEL_DATASET})')
    plt.grid(alpha=0.3)
    plt.savefig(RESULTS_DIR / f'{LOW_LABEL_DATASET}_low_label_curve.png', dpi=150, bbox_inches='tight')
    plt.show()

# Distillation gain histogram
if 'distill_df' in globals() and len(distill_df) > 0:
    plt.figure(figsize=(6, 4))
    gains = distill_df['distill_gain'].values * 100
    plt.hist(gains, bins=min(8, max(3, len(gains))), edgecolor='black')
    plt.xlabel('Distillation gain (%)')
    plt.ylabel('Count')
    plt.title(f'TinyGAT distillation gain ({DISTILL_DATASET})')
    plt.grid(alpha=0.2)
    plt.savefig(RESULTS_DIR / f'{DISTILL_DATASET}_distillation_gain.png', dpi=150, bbox_inches='tight')
    plt.show()

# Failure confidence distribution
if 'failure_df' in globals() and len(failure_df) > 0:
    plt.figure(figsize=(6, 4))
    plt.hist(failure_df['confidence'].values, bins=10, edgecolor='black')
    plt.xlabel('Prediction confidence (wrong test nodes)')
    plt.ylabel('Count')
    plt.title(f'Failure-case confidence ({FAIL_DATASET}, seed={FAIL_SEED})')
    plt.grid(alpha=0.2)
    plt.savefig(RESULTS_DIR / f'{FAIL_DATASET}_failure_confidence.png', dpi=150, bbox_inches='tight')
    plt.show()

print('Figures saved to', RESULTS_DIR)
